In this file the electrodes are cut out and saved

In [ ]:
#Needed liberary's
import customtkinter as ctk
from skimage import io, draw
from skimage.registration import phase_cross_correlation
import scipy.ndimage as ndi
import numpy as np
from PIL import Image
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
import matplotlib.pyplot as plt
import os

class main(ctk.CTk):
    def __init__(self):
        super().__init__()

        self.title("Electrode cutter")
        self.geometry("600x400")    #Sets the area that is shown in the interface
        self.map_path = r"C:\Users\manou\OneDrive\Documenten\HBO\MEA\Omni-foto's\Axion data\dag_1\A1"
        self.out_path = r"C:\Users\manou\OneDrive\Documenten\HBO\MEA\Omni-foto's\Electrodes\Axion\A1"
        self.image_paths = os.listdir(self.map_path)
        self.current_img = self.image_paths[0]
        self.image_number = 0
        self.image = io.imread((self.map_path + "/" + self.current_img))
        self.reference_image = self.image.copy()  # ── save first image as reference
        self.mask = None
        self.point = None
        self.electrodes = []
        self.image_cutoff = 0.2
        self.arm_length = 244

        self.protocol("WM_DELETE_WINDOW", self.on_close)
        ctk.set_appearance_mode("dark")
        ctk.set_default_color_theme("blue")

        self.setup_ui()

    def setup_ui(self):
        self.image_frame = ctk.CTkFrame(master=self, width=600)
        self.image_frame.pack(side="left", fill="both", expand=True, padx=10, pady=10)
        
        self.control_frame = ctk.CTkFrame(master=self, width=200)
        self.control_frame.pack(side="right", fill="y", padx=10, pady=10)
        
        self.setup_matplotlib_canvas()
        self.setup_control_buttons()
        
        self.status_label = ctk.CTkLabel(
            self.control_frame, 
            text="Vul coordinaten in of klik op de afbeelding",
            wraplength=180
        )
        self.status_label.pack(pady=10)
        
    def setup_matplotlib_canvas(self):
        self.fig, self.ax = plt.subplots(figsize=(6, 4), dpi=100)
        self.fig.patch.set_facecolor('#2b2b2b')
        self.ax.set_facecolor('#2b2b2b')
        
        self.cut_image()
        self.ax.imshow(self.image, cmap="gray")
        self.ax.set_title(f"file: {self.current_img}. {self.image_number} out of {len(self.image_paths)}\n Klik elektrode linksboven", color='white')
        self.ax.axis('off')
        
        self.canvas = FigureCanvasTkAgg(self.fig, master=self.image_frame)
        self.canvas.draw()
        self.canvas.get_tk_widget().pack(fill="both", expand=True, padx=5, pady=5)
        
        self.canvas.mpl_connect('button_press_event', self.on_image_click)
        
    def setup_control_buttons(self):
        self.point_title = ctk.CTkLabel(self.control_frame, text="── Electrode Point ──")
        self.point_title.pack(pady=(10,0), padx=10)

        self.x_label = ctk.CTkLabel(self.control_frame, text="X coordinate:")
        self.x_label.pack(pady=(5,0), padx=10)
        self.x_entry = ctk.CTkEntry(self.control_frame, placeholder_text="Enter X")
        self.x_entry.pack(pady=(0,5), padx=10, fill="x")

        self.y_label = ctk.CTkLabel(self.control_frame, text="Y coordinate:")
        self.y_label.pack(pady=(0,0), padx=10)
        self.y_entry = ctk.CTkEntry(self.control_frame, placeholder_text="Enter Y")
        self.y_entry.pack(pady=(0,5), padx=10, fill="x")

        self.set_point_btn = ctk.CTkButton(
            self.control_frame,
            text="Set Point",
            command=self.set_point_from_input,
            height=40
        )
        self.set_point_btn.pack(pady=5, padx=10, fill="x")

        self.remove_btn = ctk.CTkButton(
            self.control_frame,
            text="Remove point",
            command=self.remove_point,
            height=40
        )
        self.remove_btn.pack(pady=5, padx=10, fill="x")
        
        self.border_btn = ctk.CTkButton(
            self.control_frame,
            text="Show borders",
            command=self.show_border,
            height=40
        )
        self.border_btn.pack(pady=5, padx=10, fill="x")

        self.export_btn = ctk.CTkButton(
            self.control_frame,
            text="Export and Next",
            command=self.export_and_next,
            height=40
        )
        self.export_btn.pack(pady=5, padx=10, fill="x")

    def cut_image(self):
        h, w = self.image.shape
        cx = w//2
        cy = h//2
        dx = w * self.image_cutoff
        dy = h * self.image_cutoff
        x_start = round(cx - dx)
        x_end = round(cx + dx)
        y_start = round(cy - dy)
        y_end = round(cy + dy)
        self.image = self.image[y_start:y_end, x_start:x_end]
        self.mask = np.zeros(self.image.shape + (4,))
        self.mask[:, :, 3] = 0

    def reload_image(self):
        self.ax.clear()
        self.ax.imshow(self.image, cmap='gray')
        self.ax.imshow(self.mask)
        self.ax.set_title(f"file: {self.current_img}\n Klik elektrode linksboven", color='white')
        self.ax.axis('off')
        self.canvas.draw()

    def set_point_from_input(self):
        try:
            x = int(self.x_entry.get())
            y = int(self.y_entry.get())
        except ValueError:
            self.status_label.configure(text="Ongeldige invoer. Vul gehele getallen in.")
            return

        self.remove_point()
        self.point = (x, y)

        rr, cc = draw.disk((y, x), radius=40, shape=self.image.shape)
        self.mask[rr, cc] = [1, 0, 0, 1]
        self.reload_image()

        self.status_label.configure(text=f"Punt ingesteld: ({x}, {y})")
        print(f"Punt ingesteld via invoer: ({x}, {y})")
         
    def on_image_click(self, event):
        if event.xdata is not None and event.ydata is not None:
            self.remove_point()
            x, y = int(event.xdata), int(event.ydata)
            self.point = (x, y)
            
            rr, cc = draw.disk((y, x), radius=40, shape=self.image.shape)
            self.mask[rr, cc] = [1, 0, 0, 1]
            self.reload_image()
            
            self.status_label.configure(text=f"Punt {self.point}")
            print(f'Punt: ({x}, {y})')

    def remove_point(self):
        self.point = None
        self.mask = np.zeros(self.image.shape + (4,))
        self.reload_image()
        self.status_label.configure(text="Selectie gereset. Klik opnieuw.")
        
    def export(self):
        if not self.point:
            self.status_label.configure(text="Punt niet geselecteerd!")
            return

        if not os.path.isdir(self.out_path):
            output_dir = f"{self.map_path}/output"
            os.makedirs(output_dir)
            
        try:
            if not self.electrodes or len(self.electrodes) == 0:
                self.calc_electrodes()

            i = 0
            for row in range(1, 5):
                for col in range(1, 5):
                    electrode = self.electrodes[i]
                    ex, ey = int(electrode[0]), int(electrode[1])
                    print(f"splitting electrode: {electrode}")

                    if row == 4 and col == 4:
                        half = self.arm_length
                    else:
                        half = self.arm_length // 2

                    xstart = ex - half
                    xend   = ex + half
                    ystart = ey - half
                    yend   = ey + half

                    temp_image = self.image[ystart:yend, xstart:xend]
                    filename = f"{self.current_img.split('.')[0]}_r{5 - row}k{col}"
                    filepath = f"{self.out_path}/{filename}.jpeg"
                    io.imsave(filepath, temp_image)
                    print(f"File made: {filename}")
                    i += 1

        except Exception as e:
            self.status_label.configure(text=f"Fout bij opslaan: {str(e)}")
            print(f"Export error: {e}")

    def next(self):
        self.image_number += 1
        if self.image_number >= len(self.image_paths):
            self.status_label.configure(text="Geen afbeeldingen meer!")
            print("No more images")
            return

        self.current_img = self.image_paths[self.image_number]
        self.image = io.imread((self.map_path + "/" + self.current_img))

        # ── find and correct shift relative to first image ──
        shift, _, _ = phase_cross_correlation(
            self.reference_image,
            self.image,
            upsample_factor=10
        )
        shift_y, shift_x = int(round(shift[0])), int(round(shift[1]))
        print(f"Correcting shift: x={shift_x}, y={shift_y}")
        self.image = ndi.shift(self.image, shift=(shift_y, shift_x), mode='nearest')

        # reset
        self.point = None
        self.electrodes = []
        self.cut_image()
        self.reload_image()
        self.status_label.configure(text="Nieuwe afbeelding geladen.")

    def export_and_next(self):
        self.export()
        self.next()

    def calc_electrodes(self):
        self.electrodes = []
        for row in range(4):
            for column in range(4):
                if row == 3 and column == 3:
                    y = self.point[1] + (row * self.arm_length) + (3.5 * (self.arm_length / 8))
                    x = self.point[0] + (column * self.arm_length) + (3 * (self.arm_length / 8))
                else:
                    y = self.point[1] + row * self.arm_length
                    x = self.point[0] + column * self.arm_length

                self.electrodes.append((int(x), int(y)))
                print(f"Electrode row={row} col={column}: x={int(x)}, y={int(y)}")

        print(f"Total electrodes calculated: {len(self.electrodes)}")

    def show_border(self):
        print("Showing borders")
        thickness = 3
        self.calc_electrodes()
        counter = 1

        for electrode in self.electrodes:
            ex, ey = int(electrode[0]), int(electrode[1])
            if counter == 16:
                half = self.arm_length
            else:
                half = self.arm_length // 2

            for i in range(thickness):
                start = (ey - half + i, ex - half + i)
                end   = (ey + half - i, ex + half - i)
                rr, cc = draw.rectangle_perimeter(start, end, shape=self.image.shape)
                self.mask[rr, cc] = [1, 0, 0, 1]

            counter += 1

        self.reload_image()
    
    def on_close(self):
        plt.close(self.fig)
        self.image_frame.destroy()
        self.destroy()

if __name__ == "__main__":
    app = main()
    app.mainloop()